In [1]:
import pathlib
import matplotlib.pyplot as plt
import numpy as np
from refig import refig

from zfish.invariants import compute_invariants, select_invariants
from zfish.graphs import GraphUtils, load_all_graphs_from_edgelists

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "Roboto"


/Users/mateljk1/Documents/projects/aplbrain/zebrafish-pencilsdown/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CONTACTOME_EDGELIST_PATH = pathlib.Path("../data/graphs/contactomes/")
USING_RUSTWORKX = True


In [3]:
graphs = load_all_graphs_from_edgelists(
    CONTACTOME_EDGELIST_PATH, rustworkx=USING_RUSTWORKX
)
print(graphs.keys())


ValueError: too many values to unpack (expected 2)

In [ ]:

graphs = {
    identifier: {
        "graph": graph,
        "undirected_graph": GraphUtils.make_undirected(graph),
        "lcc_graph": GraphUtils.make_lcc(graph),
        "lwcc_nodes": GraphUtils.make_lwcc_nodes(graph),
    }
    for identifier, graph in graphs.items()
}


In [ ]:
INVARIANTS = select_invariants("contactome")

invariants_per_graph = compute_invariants(
    graphs,
    "contactome",
    INVARIANTS,
    parallel=-1,
    progress=True,
    use_rustworkx=USING_RUSTWORKX,
)


In [ ]:
with open("invariants_per_contactome.json", "w") as f:
    import json

    json.dump(invariants_per_graph, f, indent=2)


In [ ]:
with plt.style.context("bmh"):
    fig, axs = plt.subplots(
        len(INVARIANTS), 1, figsize=(3, 2 * len(INVARIANTS)), dpi=150, sharex=True
    )
    for ax, (invariant_key, invariant_spec) in zip(axs, INVARIANTS.items()):
        ylabel = invariant_spec.ylabel
        graph_ids = list(invariants_per_graph[invariant_key].keys())
        values = list(invariants_per_graph[invariant_key].values())
        ax.plot(graph_ids, values, marker="o")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel}")
        ax.grid(True)
    plt.tight_layout()
    ax.set_xlabel("Graph Identifier")
    refig.savefig("invariants_per_graph.svg")
    plt.show()


In [ ]:
with plt.style.context("seaborn-v0_8-poster"):
    fig, axs = plt.subplots(
        len(INVARIANTS), 1, figsize=(6, 4 * len(INVARIANTS)), dpi=150, sharex=True
    )
    for ax, (invariant_key, invariant_spec) in zip(axs, INVARIANTS.items()):
        ylabel = invariant_spec.ylabel
        graph_ids = list(invariants_per_graph[invariant_key].keys())
        values = np.array(list(invariants_per_graph[invariant_key].values()))
        zscores = (values - np.mean(values)) / np.std(values)
        ax.plot(graph_ids, zscores, marker="o")
        ax.set_ylabel(f"Z-score of {ylabel}")
        ax.set_title(f"Z-score of {ylabel}")
        ax.grid(True)
    plt.tight_layout()
    ax.set_xlabel("Graph Identifier")
    refig.savefig("invariants_per_graph_zscores.svg")
    plt.show()
